In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier


In [2]:
df_train = pd.read_csv('train.csv')
df_test  = pd.read_csv('test.csv')


In [3]:
test_ids = df_test['event_id'].copy()


In [4]:
drop_cols = ['event', 'time_to_hit_hours', 'event_id']

X = df_train.drop(columns=drop_cols)
X_test = df_test.drop(columns=drop_cols, errors='ignore')


In [5]:
assert list(X.columns) == list(X_test.columns)


In [6]:
def train_rf_for_horizon(hours):
    """
    Обучает RandomForest для события <= hours
    """
    y = (
        (df_train['event'] == 1) &
        (df_train['time_to_hit_hours'] <= hours)
    ).astype(int)

    rf = RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=2,
        min_samples_split=5,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )

    rf.fit(X, y)
    return rf


In [7]:
rf_12h = train_rf_for_horizon(12)
rf_24h = train_rf_for_horizon(24)
rf_48h = train_rf_for_horizon(48)
rf_72h = train_rf_for_horizon(72)


In [8]:
prob_12h = rf_12h.predict_proba(X_test)[:, 1]
prob_24h = rf_24h.predict_proba(X_test)[:, 1]
prob_48h = rf_48h.predict_proba(X_test)[:, 1]
prob_72h = rf_72h.predict_proba(X_test)[:, 1]


In [9]:
prob_24h = np.maximum(prob_24h, prob_12h)
prob_48h = np.maximum(prob_48h, prob_24h)
prob_72h = np.maximum(prob_72h, prob_48h)


In [10]:
df_sub = pd.DataFrame({
    'event_id': test_ids,
    'prob_12h': prob_12h,
    'prob_24h': prob_24h,
    'prob_48h': prob_48h,
    'prob_72h': prob_72h
})

df_sub.to_csv('submission_aveli_lavy.csv', index=False)


In [11]:
df_sub

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.082850,0.111785,0.111785,0.126198
1,13353600,0.486677,0.817789,0.817789,0.817789
2,13942327,0.062885,0.062885,0.086214,0.099262
3,16112781,0.693814,0.849916,0.849916,0.849916
4,17132808,0.585668,0.585668,0.585668,0.585668
...,...,...,...,...,...
90,94627327,0.247940,0.247940,0.247940,0.250647
91,96570675,0.055625,0.061147,0.061147,0.061147
92,97225766,0.081576,0.111607,0.112365,0.113475
93,98446281,0.238780,0.238780,0.238780,0.238780
